In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/email-dataset-categorised-as-spam-or-ham/spam.csv


In [2]:
#import libraries

import numpy as np
import pandas as pd
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
#load the data
df= pd.read_csv("/kaggle/input/email-dataset-categorised-as-spam-or-ham/spam.csv", encoding='latin1')

# mapping spam as 1 and ham as 0
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

df.sample(10)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
4797,0,Just come home. I don't want u to be miserable,NaN,NaN,NaN
3232,0,Height of recycling: Read twice- People spend ...,NaN,NaN,NaN
169,0,Yes :)it completely in out of form:)clark also...,NaN,NaN,NaN
3816,0,This is my number by vivek..,NaN,NaN,NaN
3858,1,Win the newest åÒHarry Potter and the Order of...,NaN,NaN,NaN
3692,0,I was about to do it when i texted. I finished...,NaN,NaN,NaN
918,0,Hey you gave them your photo when you register...,NaN,NaN,NaN
1440,0,Cool breeze... Bright sun... Fresh flower... T...,NaN,NaN,NaN
3714,0,"I am late,so call you tomorrow morning.take ca...",NaN,NaN,NaN
432,1,Congrats! Nokia 3650 video camera phone is you...,NaN,NaN,NaN


In [4]:
#Feature selection

df = df[['v1', 'v2']]

# rename columns (best practice)
df.columns = ['label', 'text']

# define input and output
X = df['text']
Y = df['label']

In [5]:
df.columns

Index(['label', 'text'], dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   int64 
 1   text    5572 non-null   object
dtypes: int64(1), object(1)
memory usage: 87.2+ KB


In [7]:
df.isna().sum()

label    0
text     0
dtype: int64

In [8]:

from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

#stratify keeps the ratio of spam and not spam equal

#no need of Scaling

In [9]:
# Cleaning the text (improves accuracy)

def clean_text_series(text_series):
    return text_series.apply(
        lambda text: re.sub(
            r'\s+',
            ' ',
            re.sub(r'[^a-zA-Z]', ' ', text.lower())
        )
    )

In [10]:
# converting text into numbers (ML models cannot read text.You must convert text into vectors.)
#this method doesnt flag common words like 'the, a, an'
from sklearn.feature_extraction.text import TfidfVectorizer


#building pipeline
pipeline = Pipeline([
    ('cleaner', FunctionTransformer(clean_text_series)),
    ('tfidf', TfidfVectorizer(max_features=3000)),
    ('model', MultinomialNB())
])



In [11]:
# training the model

pipeline.fit(X_train, Y_train)

Pipeline(steps=[('cleaner',
                 FunctionTransformer(func=<function clean_text_series at 0x78abb501e2a0>)),
                ('tfidf', TfidfVectorizer(max_features=3000)),
                ('model', MultinomialNB())])

In [12]:
Y_pred = pipeline.predict(X_test)
Y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [13]:
# Evaluate the model
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(Y_test, Y_pred))


Accuracy: 0.9713004484304932


In [14]:
pickle.dump(pipeline, open("spam_pipeline.pkl", "wb"))